# Day 2 Intro: Stable-Baselines3 — The PyTorch of RL 🚀

On Day 1 you coded Q-Learning and DQN from scratch. That gave you a solid understanding of *how* these algorithms work internally.

On Day 2 we shift to the **RL Engineer** perspective: using production-quality libraries to solve real problems efficiently.

**[Stable-Baselines3 (SB3)](https://stable-baselines3.readthedocs.io/)** is the go-to RL library in industry and research. It provides:
- Clean, tested implementations of A2C, PPO, SAC, TD3, DDPG, DQN
- Works with any Gymnasium-compatible environment
- Built on PyTorch with vectorised environments and TensorBoard support

This short notebook shows you the entire workflow in ~20 lines of code.

## Install dependencies

In [ ]:
!pip install stable-baselines3[extra] gymnasium --quiet

## The full SB3 workflow — 20 lines

This is all you need to train, evaluate, and save a PPO agent:

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

# 1. Create environment
env = gym.make("CartPole-v1")

# 2. Instantiate the agent — 1 line!
model = PPO("MlpPolicy", env, verbose=1)

# 3. Train
model.learn(total_timesteps=50_000)

# 4. Evaluate
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"Mean reward: {mean_reward:.1f} ± {std_reward:.1f}")

# 5. Save / load
model.save("ppo_cartpole")
loaded_model = PPO.load("ppo_cartpole")

## What just happened?

| Step | What SB3 did for you |
|---|---|
| `PPO("MlpPolicy", env)` | Built the actor + critic networks, set all default hyperparameters |
| `model.learn(50_000)` | Ran the full PPO loop: collect rollouts → compute GAE advantages → update with clipped surrogate loss |
| `evaluate_policy(...)` | Ran 10 evaluation episodes in a separate env (no training bias) |
| `model.save(...)` | Saved weights + hyperparameters to disk |

Compare this to the 300+ lines of PPO code you saw on Day 1. Same algorithm. SB3 hides the boilerplate and lets you focus on the **problem**.

## Switching algorithms — just change the class name

In [ ]:
from stable_baselines3 import A2C, DQN

# A2C on the same environment
model_a2c = A2C("MlpPolicy", env, verbose=0)
model_a2c.learn(total_timesteps=50_000)
mean_a2c, _ = evaluate_policy(model_a2c, env, n_eval_episodes=10)

# DQN on the same environment
model_dqn = DQN("MlpPolicy", env, verbose=0)
model_dqn.learn(total_timesteps=50_000)
mean_dqn, _ = evaluate_policy(model_dqn, env, n_eval_episodes=10)

print(f"PPO: {mean_reward:.1f}")
print(f"A2C: {mean_a2c:.1f}")
print(f"DQN: {mean_dqn:.1f}")

## 💡 Key SB3 concepts you'll use today

**Policies:**
- `"MlpPolicy"` — fully connected network, for flat observation vectors
- `"MultiInputPolicy"` — for dict observation spaces (used with robotics envs)
- `"CnnPolicy"` — for image observations

**Vectorised environments:** SB3 can run N environments in parallel to collect experience faster:
```python
from stable_baselines3.common.env_util import make_vec_env
vec_env = make_vec_env("CartPole-v1", n_envs=4)
```

**Callbacks** let you log metrics, save checkpoints, and stop training early:
```python
from stable_baselines3.common.callbacks import EvalCallback
callback = EvalCallback(eval_env, best_model_save_path="./logs/")
model.learn(total_timesteps=100_000, callback=callback)
```

You'll use all of these in the next two notebooks. Let's go! 🔥